# 17. Simulation — Banking Scenario Analysis

## Business Case

Banks operate under uncertainty.

Management may ask:

- What happens if default rates increase?
- What if interest rates change?
- What if operating costs rise?
- What if economic conditions improve?
- How much profit could be lost under stress?
- Which loan segments are most sensitive?

These questions are not purely prediction problems.

They are **scenario analysis and simulation** problems.

The objective is to understand:

> **How does the banking portfolio behave under different assumptions about the future?**

This notebook demonstrates a practical end-to-end simulation workflow using a synthetic loan portfolio.

## 1. Simulation vs Prediction vs Optimization

### Prediction

```text
What is likely to happen?
```

Example:

```text
Probability of default = 4.2%
```

### Optimization

```text
What action should we take?
```

Example:

```text
Allocate 300 collection agents
```

### Simulation

```text
What could happen under different assumptions?
```

Example:

```text
PD +25%
LGD +10%
Interest rate +50 bps
Operating cost +8%
```

Simulation explores possible outcomes under changing assumptions.

## 2. Banking Scenario Analysis

A scenario is a coherent set of assumptions.

Example:

### Base

```text
Normal conditions
```

### Optimistic

```text
Lower PD
Lower LGD
Slightly lower operating cost
```

### Mild Stress

```text
Higher PD
Higher LGD
Higher operating cost
```

### Severe Stress

```text
Significantly higher PD
Higher LGD
Higher operating cost
Higher interest-rate environment
```

The goal is not to predict which scenario will happen.

The goal is to understand the **impact if the assumptions occur**.

## 3. Monte Carlo Simulation

Scenario analysis can be deterministic:

```text
Base
Stress
Severe Stress
```

Monte Carlo simulation goes further.

Instead of one fixed value, uncertain variables are sampled repeatedly:

```text
PD
LGD
Interest Rate
Operating Cost
        ↓
Random Draw
        ↓
Portfolio Profit
        ↓
Repeat thousands of times
        ↓
Distribution of Outcomes
```

This allows us to estimate:

- expected profit,
- downside risk,
- percentiles,
- probability of loss.

## 4. Key Banking Metrics

This notebook uses simplified portfolio economics.

### Expected Loss

```text
Expected Loss = EAD × PD × LGD
```

where:

- EAD = Exposure at Default
- PD = Probability of Default
- LGD = Loss Given Default

### Expected Interest Income

```text
Interest Income = EAD × Interest Rate
```

### Simplified Profit

```text
Profit
=
Interest Income
-
Expected Loss
-
Operating Cost
```

This is a simplified educational framework, not a complete bank P&L or regulatory capital model.

## 5. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

df=pd.read_csv("bank_scenario_loan_portfolio_sample.csv")
scenarios=pd.read_csv("bank_scenario_assumptions.csv")

print("Loans:",len(df))
display(df.head())
display(scenarios)

## 6. Dataset Dictionary

In [ ]:
dictionary=pd.DataFrame({
    "Column":[
        "Loan_ID","Customer_Segment","Loan_Type",
        "Loan_Amount","Interest_Rate","Tenor_Months",
        "PD_Base","LGD_Base","Annual_Operating_Cost",
        "EAD","Expected_Loss_Base",
        "Expected_Interest_Income","Expected_Profit_Base"
    ],
    "Meaning":[
        "Loan identifier",
        "Customer segment",
        "Loan product",
        "Original loan amount",
        "Annual interest rate",
        "Loan tenor",
        "Base probability of default",
        "Base loss given default",
        "Annual operating cost",
        "Exposure at default",
        "Base expected loss",
        "Base expected interest income",
        "Base simplified expected profit"
    ]
})
display(dictionary)

## 7. Data Quality

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("Missing"))

print("Duplicate Loan IDs:",df["Loan_ID"].duplicated().sum())

display(df.describe().T)

## 8. Portfolio Overview

In [ ]:
portfolio_summary=pd.DataFrame({
    "Metric":[
        "Loans",
        "Total EAD",
        "Average Loan",
        "Expected Loss",
        "Interest Income",
        "Operating Cost",
        "Expected Profit"
    ],
    "Value":[
        len(df),
        df["EAD"].sum(),
        df["EAD"].mean(),
        df["Expected_Loss_Base"].sum(),
        df["Expected_Interest_Income"].sum(),
        df["Annual_Operating_Cost"].sum(),
        df["Expected_Profit_Base"].sum()
    ]
})

display(portfolio_summary)

## 9. Portfolio by Loan Type

In [ ]:
loan_type=(
    df.groupby("Loan_Type")
      .agg(
          Loans=("Loan_ID","count"),
          EAD=("EAD","sum"),
          Expected_Loss=("Expected_Loss_Base","sum"),
          Profit=("Expected_Profit_Base","sum")
      )
      .reset_index()
)

display(loan_type.round(0))

In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(
    data=loan_type,
    x="Loan_Type",
    y="Profit"
)
plt.title("Expected Profit by Loan Type")
plt.ylabel("Expected Profit")
plt.xticks(rotation=20)
plt.show()

## 10. Portfolio by Customer Segment

In [ ]:
segment=(
    df.groupby("Customer_Segment")
      .agg(
          Loans=("Loan_ID","count"),
          EAD=("EAD","sum"),
          PD=("PD_Base","mean"),
          LGD=("LGD_Base","mean"),
          Expected_Loss=("Expected_Loss_Base","sum"),
          Profit=("Expected_Profit_Base","sum")
      )
      .reset_index()
)

display(segment.round(3))

## 11. Deterministic Scenario Analysis

First, create four predefined scenarios.

Each scenario changes:

```text
PD
LGD
Interest Rate
Operating Cost
```

For example:

```text
Stress:
PD ↑
LGD ↑
Cost ↑
Rate changes
```

Then recalculate the portfolio.

In [ ]:
def run_scenario(data, pd_multiplier, lgd_multiplier,
                  rate_shock, cost_multiplier):
    x=data.copy()

    x["PD_Scenario"]=(
        x["PD_Base"]*pd_multiplier
    ).clip(0,.99)

    x["LGD_Scenario"]=(
        x["LGD_Base"]*lgd_multiplier
    ).clip(0,.99)

    x["Rate_Scenario"]=(
        x["Interest_Rate"]+rate_shock
    ).clip(.01,.30)

    x["Operating_Cost_Scenario"]=(
        x["Annual_Operating_Cost"]*
        cost_multiplier
    )

    x["Expected_Loss"]=(
        x["EAD"]*
        x["PD_Scenario"]*
        x["LGD_Scenario"]
    )

    x["Interest_Income"]=(
        x["EAD"]*
        x["Rate_Scenario"]
    )

    x["Profit"]=(
        x["Interest_Income"]
        -
        x["Expected_Loss"]
        -
        x["Operating_Cost_Scenario"]
    )

    return x

In [ ]:
scenario_results=[]

for _,s in scenarios.iterrows():
    x=run_scenario(
        df,
        s["PD_Multiplier"],
        s["LGD_Multiplier"],
        s["Rate_Shock"],
        s["Operating_Cost_Multiplier"]
    )

    scenario_results.append({
        "Scenario":s["Scenario"],
        "Total_EAD":x["EAD"].sum(),
        "Expected_Loss":x["Expected_Loss"].sum(),
        "Interest_Income":x["Interest_Income"].sum(),
        "Operating_Cost":x["Operating_Cost_Scenario"].sum(),
        "Profit":x["Profit"].sum()
    })

scenario_results=pd.DataFrame(scenario_results)

display(scenario_results.round(0))

## 12. Scenario Profit Comparison

In [ ]:
plt.figure(figsize=(9,5))

sns.barplot(
    data=scenario_results,
    x="Scenario",
    y="Profit"
)

plt.title("Portfolio Profit Under Different Scenarios")
plt.ylabel("Profit")
plt.xticks(rotation=20)
plt.show()

## 13. Scenario Loss Comparison

In [ ]:
plt.figure(figsize=(9,5))

sns.barplot(
    data=scenario_results,
    x="Scenario",
    y="Expected_Loss"
)

plt.title("Expected Credit Loss Under Scenarios")
plt.ylabel("Expected Loss")
plt.xticks(rotation=20)
plt.show()

## 14. Scenario Impact vs Base

The important management question is often:

> How much does the portfolio deteriorate relative to the base case?

Calculate:

```text
Profit Impact
=
Scenario Profit
-
Base Profit
```

and:

```text
Loss Increase
=
Scenario Expected Loss
-
Base Expected Loss
```

In [ ]:
base_profit=scenario_results.loc[
    scenario_results["Scenario"]=="Base",
    "Profit"
].iloc[0]

base_loss=scenario_results.loc[
    scenario_results["Scenario"]=="Base",
    "Expected_Loss"
].iloc[0]

scenario_results["Profit_Impact"]=(
    scenario_results["Profit"]-base_profit
)

scenario_results["Loss_Impact"]=(
    scenario_results["Expected_Loss"]-base_loss
)

display(
    scenario_results[
        [
            "Scenario","Profit",
            "Profit_Impact",
            "Expected_Loss",
            "Loss_Impact"
        ]
    ].round(0)
)

## 15. Segment Stress Testing

Portfolio-level results can hide concentration risk.

For example:

```text
Total portfolio looks healthy
```

while:

```text
Mortgage
    ↓
large exposure
    ↓
large stress loss
```

Therefore, run scenario analysis by:

- loan type,
- customer segment,
- branch,
- geography,
- vintage,
- risk grade.

In [ ]:
stress=run_scenario(
    df,
    pd_multiplier=1.50,
    lgd_multiplier=1.15,
    rate_shock=.005,
    cost_multiplier=1.10
)

segment_stress=(
    stress.groupby("Customer_Segment")
    .agg(
        EAD=("EAD","sum"),
        Expected_Loss=("Expected_Loss","sum"),
        Profit=("Profit","sum")
    )
    .reset_index()
)

display(segment_stress.round(0))

## 16. Monte Carlo Simulation

Deterministic scenarios provide a few points.

Monte Carlo provides a distribution.

For each simulation:

```text
Random PD shock
Random LGD shock
Random rate shock
Random cost shock
        ↓
Portfolio Profit
```

Repeat:

```text
5,000 times
```

Then examine:

- mean,
- median,
- P5,
- P10,
- P25,
- P50,
- P90,
- probability of negative profit.

In [ ]:
N_SIM=5000
simulation_results=[]

for i in range(N_SIM):
    # Macro shocks with correlated direction
    macro=rng.normal(0,1)

    pd_shock=np.exp(
        rng.normal(
            loc=0.0+0.10*macro,
            scale=.12
        )
    )

    lgd_shock=np.exp(
        rng.normal(
            loc=0.0+0.07*macro,
            scale=.08
        )
    )

    rate_shock=rng.normal(
        loc=0.0,
        scale=.006
    )

    cost_multiplier=np.exp(
        rng.normal(
            loc=0.0+0.03*macro,
            scale=.04
        )
    )

    x=run_scenario(
        df,
        pd_multiplier=pd_shock,
        lgd_multiplier=lgd_shock,
        rate_shock=rate_shock,
        cost_multiplier=cost_multiplier
    )

    simulation_results.append({
        "Simulation":i+1,
        "Profit":x["Profit"].sum(),
        "Expected_Loss":x["Expected_Loss"].sum(),
        "Interest_Income":x["Interest_Income"].sum(),
        "PD_Multiplier":pd_shock,
        "LGD_Multiplier":lgd_shock,
        "Rate_Shock":rate_shock,
        "Cost_Multiplier":cost_multiplier
    })

sim=pd.DataFrame(simulation_results)

display(sim.head())

## 17. Monte Carlo Profit Distribution

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    sim["Profit"],
    bins=60,
    kde=True
)

plt.axvline(
    sim["Profit"].mean(),
    ls="--",
    label="Mean"
)

plt.title("Monte Carlo Distribution of Portfolio Profit")
plt.xlabel("Profit")
plt.legend()
plt.show()

## 18. Monte Carlo Risk Metrics

In [ ]:
risk_metrics=pd.DataFrame({
    "Metric":[
        "Mean Profit",
        "Median Profit",
        "P5 Profit",
        "P10 Profit",
        "P25 Profit",
        "P75 Profit",
        "P95 Profit",
        "Probability Profit < 0"
    ],
    "Value":[
        sim["Profit"].mean(),
        sim["Profit"].median(),
        sim["Profit"].quantile(.05),
        sim["Profit"].quantile(.10),
        sim["Profit"].quantile(.25),
        sim["Profit"].quantile(.75),
        sim["Profit"].quantile(.95),
        (sim["Profit"]<0).mean()
    ]
})

display(risk_metrics)

## 19. Value at Risk Style Interpretation

For simulation purposes:

```text
P5 Profit
```

means approximately:

> 5% of simulated outcomes are below this profit level.

This is useful as a downside scenario indicator.

It should not automatically be interpreted as regulatory VaR because the methodology and economic meaning differ.

## 20. Expected Shortfall

Another useful tail metric is:

```text
Average Profit among the worst 5% outcomes
```

This describes the severity of the downside tail, not only its threshold.

In [ ]:
p5=sim["Profit"].quantile(.05)

expected_shortfall=sim.loc[
    sim["Profit"]<=p5,
    "Profit"
].mean()

print("P5 Profit:",round(p5))
print("Expected Shortfall (worst 5% mean):",round(expected_shortfall))

## 21. Probability of Stress

Management can define a stress threshold.

For example:

```text
Stress = profit below 90% of base profit
```

Then estimate:

```text
P(Profit < Stress Threshold)
```

This converts simulation into a risk probability.

In [ ]:
stress_threshold=.90*base_profit

prob_stress=(
    sim["Profit"]<stress_threshold
).mean()

print("Base profit:",round(base_profit))
print("Stress threshold:",round(stress_threshold))
print("Probability below threshold:",
      round(prob_stress,4))

## 22. Sensitivity Analysis

Which assumptions drive profit the most?

Candidate drivers:

- PD multiplier
- LGD multiplier
- interest-rate shock
- operating-cost multiplier

We can estimate simple correlations between simulation inputs and portfolio profit.

In [ ]:
sensitivity=sim[
    [
        "PD_Multiplier",
        "LGD_Multiplier",
        "Rate_Shock",
        "Cost_Multiplier",
        "Profit"
    ]
].corr()["Profit"].drop("Profit").sort_values()

display(sensitivity.to_frame("Correlation_with_Profit"))

In [ ]:
plt.figure(figsize=(8,5))

sensitivity.sort_values().plot(
    kind="barh"
)

plt.title("Simulation Driver Sensitivity")
plt.xlabel("Correlation with Profit")
plt.show()

## 23. Two-Way Stress Test

A useful management tool is a stress matrix.

Example:

```text
                 PD Shock
              0%    +20%    +40%    +60%

LGD  0%
     +10%
     +20%
     +30%
```

Each cell contains simulated or deterministic portfolio profit.

This helps management understand combinations of adverse conditions.

In [ ]:
pd_shocks=[1.0,1.2,1.4,1.6]
lgd_shocks=[1.0,1.1,1.2,1.3]

matrix=[]

for lgd_mult in lgd_shocks:
    row=[]
    for pd_mult in pd_shocks:
        x=run_scenario(
            df,
            pd_multiplier=pd_mult,
            lgd_multiplier=lgd_mult,
            rate_shock=0,
            cost_multiplier=1
        )
        row.append(x["Profit"].sum())
    matrix.append(row)

stress_matrix=pd.DataFrame(
    matrix,
    index=[f"LGD x{v:.1f}" for v in lgd_shocks],
    columns=[f"PD x{v:.1f}" for v in pd_shocks]
)

display(stress_matrix.round(0))

In [ ]:
plt.figure(figsize=(8,5))

sns.heatmap(
    stress_matrix,
    annot=True,
    fmt=".0f"
)

plt.title("PD vs LGD Stress Matrix — Portfolio Profit")
plt.xlabel("PD Multiplier")
plt.ylabel("LGD Multiplier")
plt.show()

## 24. Segment-Level Monte Carlo

A bank may want to know:

> Which customer segment contributes most to downside risk?

The simulation can be repeated at segment level.

This supports:

- portfolio concentration analysis,
- risk appetite discussions,
- product strategy,
- credit policy decisions.

In [ ]:
segment_mc=[]

for i in range(1000):
    pd_mult=np.exp(rng.normal(0,.15))
    lgd_mult=np.exp(rng.normal(0,.10))
    rate_shock=rng.normal(0,.006)
    cost_mult=np.exp(rng.normal(0,.05))

    x=run_scenario(
        df,
        pd_mult,
        lgd_mult,
        rate_shock,
        cost_mult
    )

    g=x.groupby("Customer_Segment")["Profit"].sum()

    for seg,value in g.items():
        segment_mc.append({
            "Simulation":i+1,
            "Segment":seg,
            "Profit":value
        })

segment_mc=pd.DataFrame(segment_mc)

display(
    segment_mc.groupby("Segment")["Profit"]
    .agg(["mean","median","min","max"])
    .round(0)
)

## 25. Scenario Decision Framework

Simulation should support a decision.

Example:

```text
Scenario
   ↓
Financial Impact
   ↓
Risk Threshold
   ↓
Management Trigger
   ↓
Action
```

Possible actions:

- tighten underwriting,
- increase collection capacity,
- adjust pricing,
- revise risk appetite,
- increase provisions,
- rebalance portfolio,
- reduce exposure to vulnerable segments.

The simulation does not choose the business policy automatically.

## 26. Simulation vs Stress Testing

They are related but not identical.

### Scenario Analysis

Explores specified assumptions.

```text
"What if PD increases 30%?"
```

### Monte Carlo Simulation

Explores a distribution of uncertain assumptions.

```text
"What range of outcomes could occur?"
```

### Stress Testing

Typically focuses on severe but plausible adverse conditions and their impact on financial resilience.

In banking, formal stress testing may involve institution-specific regulatory and risk frameworks beyond this educational notebook.

## 27. Production Architecture

```text
Core Banking
     ↓
Loan Portfolio
     ↓
Risk Models
     ↓
PD / LGD / EAD
     ↓
Scenario Engine
     ↓
Simulation Engine
     ↓
Portfolio Aggregation
     ↓
Risk Dashboard
     ↓
Management Decision
```

For production:

```text
Data Versioning
+
Model Versioning
+
Scenario Versioning
+
Audit Trail
```

are important.

## 28. Scenario Governance

Every scenario should document:

- scenario name,
- assumption owner,
- effective date,
- input variables,
- shock magnitude,
- rationale,
- model version,
- data version,
- approval status.

Avoid creating undocumented scenarios that cannot be reproduced.

## 29. Common Mistakes

1. Treating scenarios as forecasts.
2. Using unrealistic assumptions without documentation.
3. Ignoring correlations between risk factors.
4. Running simulation without validating the underlying models.
5. Reporting only average outcomes.
6. Ignoring downside percentiles.
7. Ignoring portfolio concentration.
8. Treating Monte Carlo output as certainty.
9. Mixing regulatory stress testing with simple scenario analysis.
10. Failing to version assumptions.
11. Ignoring model risk.
12. Not connecting scenarios to management actions.

## 30. Final Executive Summary

### Business Question

> **How resilient is the bank's loan portfolio under different economic and business scenarios?**

### End-to-End Workflow

```text
Portfolio Data
      ↓
Base Risk & Financial Metrics
      ↓
Scenario Assumptions
      ↓
Deterministic Scenario Analysis
      ↓
Stress Testing
      ↓
Monte Carlo Simulation
      ↓
Outcome Distribution
      ↓
Tail Risk Metrics
      ↓
Sensitivity Analysis
      ↓
Management Decision
```

### Key distinction

```text
Prediction
→ What is likely to happen?

Optimization
→ What should we do?

Simulation
→ What could happen under different assumptions?
```

Simulation is particularly useful for:

- credit portfolio stress,
- liquidity scenarios,
- interest-rate scenarios,
- deposit/run-off scenarios,
- collection scenarios,
- revenue scenarios,
- operational risk scenarios,
- capital planning,
- strategic banking planning.